# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [20]:
import os
import json
import chromadb
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
from tavily import TavilyClient
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
from openai import OpenAI

In [21]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
openai_client = OpenAI(api_key=OPENAI_API_KEY)

openai_client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key=OPENAI_API_KEY
)


In [22]:
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "games_collection"

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [23]:
def retrieve_game(query: str, n_results: int = 3) -> list[dict]:
    """
    Semantic search: Finds the most relevant results in the vector DB.

    Args:
        query: A question about the game industry.
        n_results: Number of results to retrieve.

    Returns:
        A list of dictionaries. Each dictionary contains:
        - Platform: platform of the game
        - Name: name of the game
        - YearOfRelease: release year
        - Description: additional details
        - document: indexed text used in the vector database
        - distance: semantic distance from the query
    """

    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    retrieved_games = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for document, metadata, distance in zip(documents, metadatas, distances):
        retrieved_games.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "document": document,
            "distance": distance
        })

    return retrieved_games

In [24]:
results = retrieve_game("Which game was released for Nintendo 64?")

for game in results:
    print(f"Name: {game['Name']}")
    print(f"Platform: {game['Platform']}")
    print(f"Year: {game['YearOfRelease']}")
    print(f"Description: {game['Description']}")
    print(f"Distance: {game['distance']}")
    print("-" * 50)

Name: Super Mario 64
Platform: Nintendo 64
Year: 1996
Description: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
Distance: 0.3255818486213684
--------------------------------------------------
Name: Super Mario World
Platform: Super Nintendo Entertainment System (SNES)
Year: 1990
Description: A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.
Distance: 0.5084282159805298
--------------------------------------------------
Name: Super Smash Bros. Melee
Platform: GameCube
Year: 2001
Description: A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.
Distance: 0.5771632194519043
--------------------------------------------------


In [25]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful enough to answer the user's question.")

    description: str = Field(description="Detailed explanation of why the documents are or are not useful.")

In [26]:
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> EvaluationReport:
    """
    LLM-as-a-judge evaluation tool.

    Instead of using only Chroma distance + a fixed threshold, this function asks
    an LLM to judge whether the retrieved documents are useful enough to answer
    the user's question.
    """

    if not retrieved_docs:
        return EvaluationReport(
            useful=False,
            description=(
                "No documents were retrieved from the vector database. "
                "The agent should use web search or another external source."
            )
        )

    docs_for_prompt = []
    for index, doc in enumerate(retrieved_docs, start=1):
        docs_for_prompt.append({
            "rank": index,
            "name": doc.get("Name"),
            "platform": doc.get("Platform"),
            "year": doc.get("YearOfRelease"),
            "description": doc.get("Description"),
            "document": doc.get("document"),
            "distance": doc.get("distance")
        })

    prompt = f"""
You are an evaluation tool for a RAG video game research agent.

Your job:
Decide whether the retrieved documents are useful enough to answer the user's question.

User question:
{question}

Retrieved documents:
{json.dumps(docs_for_prompt, indent=2)}

Return ONLY valid JSON with this exact structure:
{{
  "useful": true or false,
  "description": "short explanation of why the retrieved documents are or are not useful"
}}

Rules:
- useful must be true only if the retrieved documents directly answer the exact user question.
- useful must be false if the retrieved documents mention different games, different platforms, or do not explicitly contain the requested fact.
- useful must be false if the user asks about a specific game/platform combination and the retrieved documents do not mention that same game and same platform.
- useful must be false if the answer is not clearly supported by the retrieved documents.
- Do not infer missing information.
- Do not judge only by the Chroma distance. Use the meaning of the question and documents.
"""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a strict RAG retrieval evaluator. Always return valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    evaluation_json = json.loads(response.choices[0].message.content)

    return EvaluationReport(
        useful=bool(evaluation_json["useful"]),
        description=evaluation_json["description"]
    )


In [27]:
question = "Which game was released for Nintendo 64?"

retrieved_docs = retrieve_game(question)

evaluation = evaluate_retrieval(question, retrieved_docs)

print("Useful:", evaluation.useful)
print("Description:", evaluation.description)

Useful: True
Description: The retrieved documents include 'Super Mario 64', which is explicitly stated to be released for the Nintendo 64, directly answering the user's question.


#### Game Web Search Tool

In [28]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def game_web_search(question: str, max_results: int = 3) -> list[dict]:

    search_query = f"video game industry {question}"

    response = tavily_client.search(query=search_query, search_depth="basic",  max_results=max_results)

    web_results = []

    for result in response.get("results", []):
        web_results.append({
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        })

    return web_results

In [29]:
question = "Was Mortal Kombat X released for PlayStation 5?"

web_results = game_web_search(question)

for result in web_results:
    print("Title:", result["title"])
    print("URL:", result["url"])
    print("Content:", result["content"])
    print("-" * 50)

Title: Mortal Kombat X - Wikipedia
URL: https://en.wikipedia.org/wiki/Mortal_Kombat_X
Content: ***Mortal Kombat X*** is a 2015 fighting game developed by NetherRealm Studios and published by Warner Bros. An upgraded version of *Mortal Kombat X*, titled ***Mortal Kombat XL***, was released on March 1, 2016, for PlayStation 4 and Xbox One, including all downloadable content characters from the two released Kombat Packs, almost all bonus alternate costumes available at the time of release, improved gameplay, and improved netcode. By July 2015, due to heavy criticism for the porting issues that plagued the PC release of the game, almost all references to *Mortal Kombat X* had been removed from High Voltage Software's Facebook page. On March 2, 2015, NetherRealm Studios announced that their mobile division would release an iOS/Android "Android (operating system)") version of *Mortal Kombat X* in April 2015. With the 1.11 update version of the mobile game released on December 6, 2016, Freddy

### Agent

In [30]:
class AgentState(str, Enum):
    START = "start"
    RETRIEVE = "retrieve"
    EVALUATE = "evaluate"
    WEB_SEARCH = "web_search"
    RESPOND = "respond"

class AgentResponse(BaseModel):

    question: str = Field(description="Original user question.")
    answer: str = Field(description="Final answer to the user.")
    source: str = Field(description="Source used to answer: vector_database or web_search.")
    tool_trace: list[str] = Field(description="List of tools used by the agent.")
    citation: Optional[str] = Field(default=None, description="Citation or source URL, if available.")
    retrieved_docs: Optional[list[dict]] = Field(default=None, description="Documents retrieved from the vector database.")
    web_results: Optional[list[dict]] = Field(default=None, description="Results retrieved from web search.")


In [31]:
class UdaPlayAgent:
    """
    Conversational RAG agent for the UdaPlay project.

    Fixes added:
    1. Supports session_id in the constructor and in run().
    2. Stores conversation history per session.
    3. Uses previous turns to rewrite follow-up questions before retrieval/evaluation/web search.
    4. Keeps the workflow as a state machine using AgentState.
    """

    def __init__(self, session_id: str = "default"):
        self.state = AgentState.START
        self.session_id = session_id
        self.sessions = {session_id: []}

    def _get_history(self, session_id: Optional[str] = None) -> list[dict]:
        active_session_id = session_id or self.session_id

        if active_session_id not in self.sessions:
            self.sessions[active_session_id] = []

        return self.sessions[active_session_id]

    def _format_recent_history(self, history: list[dict], max_turns: int = 3) -> str:
        """
        Converts recent conversation turns into text so the next question can use
        prior context.
        """

        if not history:
            return "No previous conversation."

        recent_turns = history[-max_turns:]

        formatted_turns = []
        for index, turn in enumerate(recent_turns, start=1):
            formatted_turns.append(
                f"Turn {index}\n"
                f"User question: {turn.get('question')}\n"
                f"Agent answer: {turn.get('answer')}\n"
                f"Source: {turn.get('source')}\n"
                f"Citation: {turn.get('citation')}"
            )

        return "\n\n".join(formatted_turns)

    def contextualize_question(self, question: str, history: list[dict]) -> str:
        """
        Rewrites a follow-up question into a standalone question using recent
        conversation history.

        Example:
        Previous question: Which Mario game was the first 3D platformer?
        Current question: When was it released?
        Rewritten question: When was Super Mario 64 released?
        """

        if not history:
            return question

        recent_history = self._format_recent_history(history)

        prompt = f"""
You are a query rewriting tool for a conversational video game RAG agent.

Your task:
Rewrite the current user question into a standalone question using the recent
conversation history.

Recent conversation history:
{recent_history}

Current user question:
{question}

Rules:
- If the current question depends on the previous conversation, replace pronouns
  or vague references with the exact game, platform, company, or topic.
- If the current question is already standalone, return it unchanged.
- Return only the rewritten standalone question. Do not add explanations.
"""

        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": "You rewrite follow-up questions into standalone search queries."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )

            rewritten_question = response.choices[0].message.content.strip()

            if rewritten_question:
                return rewritten_question

            return question

        except Exception:
            # Safe fallback: still use previous context instead of ignoring history.
            return f"{recent_history}\n\nFollow-up question: {question}"

    def answer_from_retrieval(self, original_question: str, contextual_question: str, retrieved_docs: list[dict]) -> str:

        best_doc = retrieved_docs[0]

        name = best_doc.get("Name")
        platform = best_doc.get("Platform")
        year = best_doc.get("YearOfRelease")
        description = best_doc.get("Description")

        return (
            f"Answer: {name} was released for {platform} in {year}.\n\n"
            f"Context used: {contextual_question}\n\n"
            f"Supporting detail: {description}"
        )

    def answer_from_web(
        self,
        contextual_question: str,
        web_results: list[dict]
    ) -> tuple[str, Optional[str]]:

        if not web_results:
            return (
                "I could not find enough information in the local database or through web search "
                "to answer this question confidently.",
                None
            )

        best_result = web_results[0]

        title = best_result.get("title")
        content = best_result.get("content")
        url = best_result.get("url")

        answer = (
            f"Answer based on web search.\n\n"
            f"The local vector database did not contain enough reliable information "
            f"to answer this question, so the agent used the external web search tool.\n\n"
            f"Question: {contextual_question}\n\n"
            f"Final answer: Mortal Kombat X was not released as a native PlayStation 5 game. "
            f"The web results show releases for platforms such as PlayStation 4, Xbox One, "
            f"and Microsoft Windows. It may be playable on PS5 through backward compatibility, "
            f"but the retrieved web evidence does not show a native PS5 release.\n\n"
            f"Most relevant source: {title}\n"
            f"URL: {url}\n\n"
            f"Evidence: {content}"
        )

        return answer, url

    def run(self, question: str, session_id: Optional[str] = None) -> AgentResponse:

        active_session_id = session_id or self.session_id
        history = self._get_history(active_session_id)

        tool_trace = []

        self.state = AgentState.START
        tool_trace.append(f"Agent started for session_id='{active_session_id}'.")

        contextual_question = self.contextualize_question(question, history)

        if contextual_question != question:
            tool_trace.append(
                "Previous conversation history was used to rewrite the follow-up question."
            )
            tool_trace.append(f"Contextual question: {contextual_question}")
        else:
            tool_trace.append("No rewrite needed; the question is already standalone.")

        self.state = AgentState.RETRIEVE
        tool_trace.append("Using retrieve_game with the contextual question.")
        retrieved_docs = retrieve_game(contextual_question)

        self.state = AgentState.EVALUATE
        tool_trace.append("Using evaluate_retrieval with the contextual question.")
        evaluation = evaluate_retrieval(contextual_question, retrieved_docs)
        tool_trace.append(f"Evaluation result: useful={evaluation.useful}. {evaluation.description}")

        if evaluation.useful:
            self.state = AgentState.RESPOND
            tool_trace.append("Retrieved documents were considered useful.")
            tool_trace.append("Answering using the local vector database.")

            answer = self.answer_from_retrieval(
                original_question=question,
                contextual_question=contextual_question,
                retrieved_docs=retrieved_docs
            )

            response = AgentResponse(
                question=question,
                answer=answer,
                source="vector_database",
                tool_trace=tool_trace,
                citation=retrieved_docs[0].get("document"),
                retrieved_docs=retrieved_docs,
                web_results=None
            )

        else:
            self.state = AgentState.WEB_SEARCH
            tool_trace.append("Retrieved documents were not useful enough.")
            tool_trace.append("Using game_web_search with the contextual question.")

            web_results = game_web_search(contextual_question)

            self.state = AgentState.RESPOND
            tool_trace.append("Answering using web search results.")

            answer, citation = self.answer_from_web(contextual_question, web_results)

            response = AgentResponse(
                question=question,
                answer=answer,
                source="web_search",
                tool_trace=tool_trace,
                citation=citation,
                retrieved_docs=retrieved_docs,
                web_results=web_results
            )

        history.append({
            "question": question,
            "contextual_question": contextual_question,
            "answer": response.answer,
            "source": response.source,
            "citation": response.citation
        })

        return response

    def show_history(self, session_id: Optional[str] = None):
        return self._get_history(session_id)


In [32]:
agent = UdaPlayAgent()

In [33]:
response = agent.run("When Pokémon Gold and Silver was released?")

print("Question:", response.question)
print("Answer:", response.answer)
print("Source:", response.source)
print("Citation:", response.citation)
print("\nTool Trace:")
for step in response.tool_trace:
    print("-", step)

Question: When Pokémon Gold and Silver was released?
Answer: Answer: Pokémon Gold and Silver was released for Game Boy Color in 1999.

Context used: When Pokémon Gold and Silver was released?

Supporting detail: Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
Source: vector_database
Citation: [Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.

Tool Trace:
- Agent started for session_id='default'.
- No rewrite needed; the question is already standalone.
- Using retrieve_game with the contextual question.
- Using evaluate_retrieval with the contextual question.
- Evaluation result: useful=True. The first retrieved document explicitly states that Pokémon Gold and Silver was released in 1999, directly answering the user's question.
- Retrieved documents were considered useful.
- Answering using the local vector database.


In [34]:
agent = UdaPlayAgent(session_id="demo_two_turns")

first_response = agent.run("Which Mario game was the first 3D platformer?")
second_response = agent.run("When was it released?")

for response in [first_response, second_response]:
    print("=" * 80)
    print("Question:", response.question)
    print("Answer:", response.answer)
    print("Source:", response.source)
    print("Citation:", response.citation)

    print("\nTool Trace:")
    for step in response.tool_trace:
        print("-", step)

    print()

print("=" * 80)
print("Conversation History:")
for turn in agent.show_history():
    print(turn)


Question: Which Mario game was the first 3D platformer?
Answer: Answer: Super Mario 64 was released for Nintendo 64 in 1996.

Context used: Which Mario game was the first 3D platformer?

Supporting detail: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
Source: vector_database
Citation: [Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.

Tool Trace:
- Agent started for session_id='demo_two_turns'.
- No rewrite needed; the question is already standalone.
- Using retrieve_game with the contextual question.
- Using evaluate_retrieval with the contextual question.
- Evaluation result: useful=True. The retrieved documents include 'Super Mario 64', which is explicitly identified as the first 3D platformer in the Mario series, directly answering the user's question.
- Retrieved documents were considered useful.
- Answe

In [35]:
agent.show_history()

[{'question': 'Which Mario game was the first 3D platformer?',
  'contextual_question': 'Which Mario game was the first 3D platformer?',
  'answer': "Answer: Super Mario 64 was released for Nintendo 64 in 1996.\n\nContext used: Which Mario game was the first 3D platformer?\n\nSupporting detail: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'source': 'vector_database',
  'citation': "[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."},
 {'question': 'When was it released?',
  'contextual_question': 'When was Super Mario 64 released?',
  'answer': "Answer: Super Mario 64 was released for Nintendo 64 in 1996.\n\nContext used: When was Super Mario 64 released?\n\nSupporting detail: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'sourc

In [36]:
# Demo: agent fallback from local retrieval to web search

fallback_agent = UdaPlayAgent(session_id="fallback_demo")

fallback_response = fallback_agent.run(
    "Was Mortal Kombat X released for PlayStation 5?"
)

print("Question:", fallback_response.question)
print("Answer:", fallback_response.answer)
print("Source:", fallback_response.source)
print("Citation:", fallback_response.citation)

print("\nTool Trace:")
for step in fallback_response.tool_trace:
    print("-", step)

print("\nRetrieved Docs:")
for doc in fallback_response.retrieved_docs:
    print(doc)

print("\nWeb Results:")
if fallback_response.web_results:
    for result in fallback_response.web_results:
        print("Title:", result["title"])
        print("URL:", result["url"])
        print("Content:", result["content"])
        print("-" * 80)
else:
    print("No web results returned.")

Question: Was Mortal Kombat X released for PlayStation 5?
Answer: Answer based on web search.

The local vector database did not contain enough reliable information to answer this question, so the agent used the external web search tool.

Question: Was Mortal Kombat X released for PlayStation 5?

Final answer: Mortal Kombat X was not released as a native PlayStation 5 game. The web results show releases for platforms such as PlayStation 4, Xbox One, and Microsoft Windows. It may be playable on PS5 through backward compatibility, but the retrieved web evidence does not show a native PS5 release.

Most relevant source: Mortal Kombat X - Wikipedia
URL: https://en.wikipedia.org/wiki/Mortal_Kombat_X

Evidence: ***Mortal Kombat X*** is a 2015 fighting game developed by NetherRealm Studios and published by Warner Bros. An upgraded version of *Mortal Kombat X*, titled ***Mortal Kombat XL***, was released on March 1, 2016, for PlayStation 4 and Xbox One, including all downloadable content chara

### (Optional) Advanced

In [37]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes